# บทเรียน 10 - เอเย่นต์ AI ในการใช้งานจริง

ในบทเรียนนี้คุณจะได้เรียนรู้ **รูปแบบการใช้งานในระดับการผลิต** สำหรับเอเย่นต์ปัญญาประดิษฐ์โดยใช้ Microsoft Agent Framework (Python). เราจะครอบคลุม:

- **Observability** — เพิ่มการบันทึกเวลาและการล็อกในการโต้ตอบของเอเย่นต์
- **Evaluation** — การใช้เอเย่นต์ผู้ประเมินเพื่อให้คะแนนคุณภาพของการตอบกลับ
- **Cost management** — กลยุทธ์สำหรับการเพิ่มประสิทธิภาพโทเค็นและการเลือกโมเดล

สถานการณ์คือ **ตัวแทนการเดินทาง** ที่ช่วยผู้ใช้วางแผนการเดินทาง โดยมีการติดตามและการประเมินผลซ้อนทับอยู่ด้านบน


## การตั้งค่า


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ข้อพิจารณาสำหรับการนำไปใช้ในสภาพแวดล้อมการผลิต

การย้ายเอเจนต์ AI จากต้นแบบไปสู่การผลิตต้องให้ความสำคัญอย่างรอบคอบกับสามเสาหลัก:

1. **การสังเกตการณ์** — คุณต้องมีความสามารถในการมองเห็นว่าเอเจนต์กำลังทำอะไร ใช้เวลานานเท่าใด และเรียกใช้เครื่องมือใดบ้าง หากไม่มีการติดตามและการบันทึก การดีบักปัญหาในสภาพแวดล้อมการผลิตแทบจะเป็นไปไม่ได้

2. **การประเมินผล** — การตรวจสอบคุณภาพแบบอัตโนมัติช่วยให้มั่นใจว่าการตอบของเอเจนต์ยังคงถูกต้อง ครบถ้วน และเป็นประโยชน์ตลอดเวลา เอเจนต์ผู้ประเมินสามารถให้คะแนนการตอบตามเกณฑ์ที่กำหนดได้

3. **การจัดการค่าใช้จ่าย** — การใช้โทเค็นส่งผลโดยตรงต่อค่าใช้จ่าย ยุทธศาสตร์เช่นการเพิ่มประสิทธิภาพพรอมต์ การเลือกโมเดล และการแคชช่วยให้ควบคุมค่าใช้จ่ายได้โดยไม่ลดทอนคุณภาพ


## การสร้างตัวแทนที่สังเกตได้

เรากำหนดเครื่องมือการเดินทางและห่อการเรียกตัวแทนด้วยการจับเวลาเพื่อที่เราจะสามารถตรวจสอบความหน่วงได้ ในการใช้งานจริงคุณจะผสานรวมกับ OpenTelemetry หรือแบ็กเอนด์การติดตามที่คล้ายกัน


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## รูปแบบการประเมิน

รูปแบบที่พบบ่อยในการใช้งานจริงคือการใช้เอเจนต์ตัวที่สองเป็น **ผู้ประเมิน**. ผู้ประเมินจะให้คะแนนการตอบของเอเจนต์หลักตามเกณฑ์ที่กำหนดไว้ล่วงหน้า เช่น ความครบถ้วน ความถูกต้อง และความเป็นประโยชน์.

สิ่งนี้ช่วยให้:
- เกณฑ์การควบคุมคุณภาพอัตโนมัติก่อนที่การตอบจะถึงผู้ใช้
- การตรวจจับการถดถอยเมื่อพรอมต์หรือโมเดลเปลี่ยนแปลง
- การตรวจสอบผลการทำงานของเอเจนต์อย่างต่อเนื่องเมื่อเวลาผ่านไป


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## กลยุทธ์การจัดการต้นทุน

การควบคุมต้นทุนเป็นสิ่งสำคัญสำหรับเอเจนต์ AI ที่ใช้งานในแวดล้อมการผลิต ต่อไปนี้เป็นกลยุทธ์สำคัญ:

| กลยุทธ์ | คำอธิบาย |
|---|---|
| **การปรับแต่งพรอมต์** | ทำให้คำสั่งของระบบสั้นและกระชับ. ลบบริบทที่ซ้ำซ้อนเพื่อลดจำนวนโทเค็นนำเข้า. |
| **การเลือกโมเดล** | ใช้โมเดลที่มีขนาดเล็กและถูกกว่า (เช่น GPT-4o-mini) สำหรับงานง่าย ๆ เช่น การจำแนกหรือการสกัดข้อมูล และสงวนการใช้โมเดลขนาดใหญ่สำหรับการให้เหตุผลที่ซับซ้อน. |
| **การแคช** | เก็บผลลัพธ์จากเครื่องมือและคำค้นหาที่ใช้บ่อยในแคชเพื่อหลีกเลี่ยงการเรียก API ซ้ำซ้อน. |
| **งบประมาณโทเค็น** | ตั้งขีดจำกัด `max_tokens` เพื่อป้องกันการตอบกลับที่ยาวเกินคาด. |
| **การประมวลผลเป็นชุด** | รวมคำถามจากผู้ใช้หลายรายการเป็นการเรียก API เดียวเมื่อเป็นไปได้. |

ในทางปฏิบัติ การใช้แนวทางแบบเป็นชั้น ๆ ให้ผลดี: ส่งคำขอที่ตรงไปตรงมาไปยังโมเดลที่รวดเร็วและราคาประหยัด และยกระดับเฉพาะคำถามที่ซับซ้อนให้ไปยังโมเดลที่มีความสามารถมากกว่า.


## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

1. **เพิ่มการสังเกตการณ์** ให้กับการโต้ตอบของเอเจนต์ด้วยการจับเวลาและการบันทึก, วางรากฐานสำหรับการติดตามและการตรวจสอบ.
2. **ประเมินการตอบของเอเจนต์** โดยอัตโนมัติด้วยเอเจนต์ผู้ประเมินที่ให้คะแนนความครบถ้วน ความถูกต้อง และความเป็นประโยชน์.
3. **จัดการค่าใช้จ่าย** ผ่านการปรับแต่งพรอมต์ การเลือกโมเดล การแคช และงบประมาณโทเค็น.

รูปแบบการนำไปใช้งานเหล่านี้ช่วยให้มั่นใจว่าเอเจนต์ AI ของคุณน่าเชื่อถือ สามารถวัดผลได้ และคุ้มค่าต่อค่าใช้จ่ายเมื่อใช้งานในระดับขนาดใหญ่.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลด้วยปัญญาประดิษฐ์ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะมุ่งมั่นให้การแปลมีความถูกต้อง แต่โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำได้ เอกสารต้นฉบับในภาษาดั้งเดิมควรถือเป็นแหล่งข้อมูลที่เป็นทางการ สำหรับข้อมูลที่มีความสำคัญ ขอแนะนำให้ใช้บริการแปลโดยมนุษย์ผู้เชี่ยวชาญ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใด ๆ ที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
